In [ ]:
import pandas as pd
from tqdm import tqdm

# PATHS 
BM25_PATH = r"C:\Users\nihal\OneDrive\Desktop\NLP_Project\bm25_subset_candidates.csv"
QRELS_PATH = r"C:\Users\nihal\OneDrive\Desktop\NLP_Project\data\msmarco_passage\qrels.train.tsv"
OUTPUT_PATH = r"C:\Users\nihal\OneDrive\Desktop\NLP_Project\training_pairs.csv"

MAX_PAIRS_PER_QUERY = 5


print("Loading BM25 candidates...")
bm25_df = pd.read_csv(BM25_PATH, dtype=str)

print("Loading qrels...")
qrels_df = pd.read_csv(
    QRELS_PATH,
    sep="\t",
    names=["query_id", "unused", "passage_id", "relevance"],
    dtype=str
)

# Keep only relevant judgments
qrels_df = qrels_df[qrels_df["relevance"] == "1"]

# Build lookup for relevance
relevant_set = set(
    zip(qrels_df["query_id"], qrels_df["passage_id"])
)

print("Building training pairs...")
training_rows = []

grouped = bm25_df.groupby("query_id")

for query_id, group in tqdm(grouped):
    query_text = group.iloc[0]["query"]

    positives = []
    negatives = []

    for _, row in group.iterrows():
        key = (row["query_id"], row["passage_id"])
        if key in relevant_set:
            positives.append(row["passage"])
        else:
            negatives.append(row["passage"])

    if not positives or not negatives:
        continue

    count = 0
    for pos in positives:
        for neg in negatives:
            training_rows.append({
                "query": query_text,
                "positive_passage": pos,
                "negative_passage": neg
            })
            count += 1
            if count >= MAX_PAIRS_PER_QUERY:
                break
        if count >= MAX_PAIRS_PER_QUERY:
            break

training_df = pd.DataFrame(training_rows)
training_df.to_csv(OUTPUT_PATH, index=False)

print("Step 2 completed.")
print(f"Training pairs generated: {len(training_df)}")
print(f"Saved to: {OUTPUT_PATH}")


Loading BM25 candidates...
Loading qrels...
Building training pairs...


100%|██████████| 30000/30000 [03:47<00:00, 131.59it/s]


Step 2 completed.
Training pairs generated: 130790
Saved to: C:\Users\nihal\OneDrive\Desktop\NLP_Project\training_pairs.csv
